In [0]:
spark.sql("CREATE CATALOG IF NOT EXISTS retail_project")
spark.sql("CREATE SCHEMA IF NOT EXISTS retail_project.retail_silver")

spark.sql("USE CATALOG retail_project")
spark.sql("USE SCHEMA retail_silver")

from pyspark.sql.functions import *

### CUSTOMER

In [0]:
profile = spark.table("retail_project.retail_bronze.bronze_customers_profile")
loyalty = spark.table("retail_project.retail_bronze.bronze_customers_loyalty")
address = spark.table("retail_project.retail_bronze.bronze_customers_address")
contact = spark.table("retail_project.retail_bronze.bronze_customers_contact")

In [0]:
profile = profile.dropDuplicates(["customer_id"])
loyalty = loyalty.dropDuplicates(["customer_id"])
address = address.dropDuplicates(["customer_id"])
contact = contact.dropDuplicates(["customer_id"])

In [0]:
customer_df = (
    profile
    .join(loyalty, "customer_id", "left")
    .join(address, "customer_id", "left")
    .join(contact, "customer_id", "left")
)

In [0]:
customer_df = customer_df \
.withColumn("first_name", initcap(col("first_name"))) \
.withColumn("last_name", initcap(col("last_name"))) \
.withColumn("customer_full_name", concat_ws(" ", col("first_name"), col("last_name"))) \
.withColumn("customer_location", concat_ws(", ", col("city"), col("state"), col("country"))) \
.withColumn("customer_status", when(col("active_status") == "Y", "Active").otherwise("Inactive"))

In [0]:
customer_df.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema", "true")\
.saveAsTable("retail_project.retail_silver.customer")

### INVENTORY

In [0]:
inventory_supplier = spark.table("retail_project.retail_bronze.bronze_inventory_supplier")
inventory_cost = spark.table("retail_project.retail_bronze.bronze_inventory_cost")
inventory_stock = spark.table("retail_project.retail_bronze.bronze_inventory_stock")

In [0]:
inventory_df = (
    inventory_supplier
    .join(inventory_cost, "inventory_id", "left")
    .join(inventory_stock, "inventory_id", "left")
)

In [0]:
inventory_df = inventory_df \
.withColumn("inventory_value", col("unit_cost") * col("stock_qty")) \
.withColumn("days_between_supply", datediff(col("next_supply_date"), col("last_supply_date"))) \
.withColumn("stock_status", when(col("stock_qty") <= col("reorder_level"), "Reorder Needed").otherwise("Stock Available"))

In [0]:
inventory_df.write.format("delta").mode("overwrite").saveAsTable("retail_project.retail_silver.inventory")

### ORDERS

In [0]:
orders_items = spark.table("retail_project.retail_bronze.bronze_orders_items")
orders_main = spark.table("retail_project.retail_bronze.bronze_orders_main")
orders_promo = spark.table("retail_project.retail_bronze.bronze_orders_promo")
orders_shipping = spark.table("retail_project.retail_bronze.bronze_orders_shipping")
orders_channel = spark.table("retail_project.retail_bronze.bronze_orders_channel")

In [0]:
orders_items = orders_items.dropDuplicates(["order_id"])
orders_main = orders_main.dropDuplicates(["order_id"])
orders_promo = orders_promo.dropDuplicates(["order_id"])
orders_shipping = orders_shipping.dropDuplicates(["order_id"])
orders_channel = orders_channel.dropDuplicates(["order_id"])

In [0]:
orders_df = (
    orders_main
    .join(orders_items, "order_id", "left")
    .join(orders_promo, "order_id", "left")
    .join(orders_shipping, "order_id", "left")
    .join(orders_channel, "order_id", "left")
)

In [0]:
orders_df = orders_df \
.withColumn("calculated_order_value", col("quantity") * col("unit_price")) \
.withColumn("delivery_days", datediff(col("delivery_date"), col("dispatch_date"))) \
.withColumn("promo_applied_flag", when(col("promo_code").isNotNull(), 1).otherwise(0)) \
.withColumn("shipping_cost_per_item", col("shipping_cost") / col("quantity"))

In [0]:
orders_df.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema", "true")\
.saveAsTable("retail_project.retail_silver.orders")


### PAYMENTS

In [0]:
payments_main = spark.table("retail_project.retail_bronze.bronze_payments_main")
payment_amount = spark.table("retail_project.retail_bronze.bronze_payments_amount")
payment_gateway = spark.table("retail_project.retail_bronze.bronze_payments_gateway")

In [0]:
payments_main = payments_main.dropDuplicates(["payment_id"])
payment_amount = payment_amount.dropDuplicates(["payment_id"])
payment_gateway = payment_gateway.dropDuplicates(["payment_id"])

In [0]:
payments_df = (
    payments_main
    .join(payment_amount, "payment_id", "left")
    .join(payment_gateway, "payment_id", "left")
)

In [0]:
payments_df = payments_df \
.withColumn("total_payment_value", col("amount") + col("tax")) \
.withColumn("payment_success_flag", when(col("payment_status") == "Success", 1).otherwise(0)) \
.withColumn("fraud_risk_category",
    when(col("risk_score") >= 80, "High Risk")
    .when(col("risk_score") >= 50, "Medium Risk")
    .otherwise("Low Risk"))

In [0]:
payments_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("retail_project.retail_silver.payments")

### PRODUCTS

In [0]:
products_catalog = spark.table("retail_project.retail_bronze.bronze_products_catalog")
product_supplier = spark.table("retail_project.retail_bronze.bronze_products_supplier")
product_price = spark.table("retail_project.retail_bronze.bronze_products_price")

In [0]:
products_catalog = products_catalog.dropDuplicates(["product_id"])
product_supplier = product_supplier.dropDuplicates(["product_id"])
product_price = product_price.dropDuplicates(["product_id"])

In [0]:
products_df = (
    products_catalog
    .join(product_supplier, "product_id", "left")
    .join(product_price, "product_id", "left")
)

In [0]:
products_df = products_df \
.withColumn("discounted_price", col("price") - (col("price") * col("discount") / 100)) \
.withColumn("final_price_with_tax", col("discounted_price") + (col("discounted_price") * col("tax_rate") / 100)) \
.withColumn("product_popularity",
    when(col("rating") >= 4, "High")
    .when(col("rating") >= 3, "Medium")
    .otherwise("Low")) \
.withColumn("product_age", year(current_date()) - col("launch_year"))

In [0]:
products_df.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema", "true")\
.saveAsTable("retail_project.retail_silver.products")

### RETURNS

In [0]:
return_main = spark.table("retail_project.retail_bronze.bronze_returns_main")
return_logistics = spark.table("retail_project.retail_bronze.bronze_returns_logistics")
return_refund = spark.table("retail_project.retail_bronze.bronze_returns_refund")

In [0]:
return_main = return_main.dropDuplicates(["return_id"])
return_logistics = return_logistics.dropDuplicates(["return_id"])
return_refund = return_refund.dropDuplicates(["return_id"])

In [0]:
returns_df = (
    return_main
    .join(return_logistics, "return_id", "left")
    .join(return_refund, "return_id", "left")
)

In [0]:
returns_df = returns_df \
.withColumn("reason", trim(col("reason"))) \
.withColumn("return_processing_time", datediff(col("refund_date"), col("return_date"))) \
.withColumn("refund_completed_flag", when(col("status") == "Approved", 1).otherwise(0))

In [0]:
returns_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("retail_project.retail_silver.returns")

### SHIPMENTS

In [0]:
ship_main = spark.table("retail_project.retail_bronze.bronze_shipments_main")
ship_tracking = spark.table("retail_project.retail_bronze.bronze_shipments_tracking")
ship_cost = spark.table("retail_project.retail_bronze.bronze_shipments_cost")
ship_warehouse = spark.table("retail_project.retail_bronze.bronze_shipments_warehouse")

In [0]:
ship_main = ship_main.dropDuplicates(["shipment_id"])
ship_tracking = ship_tracking.dropDuplicates(["shipment_id"])
ship_cost = ship_cost.dropDuplicates(["shipment_id"])
ship_warehouse = ship_warehouse.dropDuplicates(["shipment_id"])

In [0]:
shipments_df = (
    ship_main
    .join(ship_tracking, "shipment_id", "left")
    .join(ship_cost, "shipment_id", "left")
    .join(ship_warehouse, "shipment_id", "left")
)

In [0]:
shipments_df = shipments_df \
.withColumn("estimated_delivery_days", col("eta_days")) \
.withColumn("transport_cost_estimate", col("distance_km") * col("cost_per_km")) \
.withColumn("delivery_delay_flag", when(col("delivery_status") == "InTransit", 1).otherwise(0))

In [0]:
shipments_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("retail_project.retail_silver.shipments")

### STORE

In [0]:
store_info = spark.table("retail_project.retail_bronze.bronze_stores_info")
store_area = spark.table("retail_project.retail_bronze.bronze_stores_area")
store_manager = spark.table("retail_project.retail_bronze.bronze_stores_manager")

In [0]:
store_info = store_info.dropDuplicates(["store_id"])
store_area = store_area.dropDuplicates(["store_id"])
store_manager = store_manager.dropDuplicates(["store_id"])

In [0]:
store_df = (
    store_info
    .join(store_area, "store_id", "left")
    .join(store_manager, "store_id", "left")
)

In [0]:
store_df = store_df \
.withColumn("store_age", year(current_date()) - col("open_year")) \
.withColumn("store_size_category",
    when(col("area_sqft") >= 4000, "Large")
    .when(col("area_sqft") >= 2000, "Medium")
    .otherwise("Small"))

In [0]:
store_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("retail_project.retail_silver.store")